# QIGOA - Stage ORACLE (dong rui ro #4 + thang tran cho Bang V / Hinh 4)

Chay `run_ceiling.py --stage all` tren `exp_ceiling.yaml`: oracle_single ⊆ oracle_interval ⊆ oracle_levelset (loai C, dung GT) + classical (A) + P5 (B). ~35 phut CPU.

**Cau hinh:** Internet ON - Accelerator **None** - Input `awsaf49/brats20-dataset-training-validation`.

Sinh tu `notebooks/qigoa_oracle.ipynb`. Moi lenh qua `sh()` -> raise -> FAILED (khong troi am tham).

## Cell 1 - clone + guard + pip

In [ ]:
# Batch-safe: moi loi -> raise -> Kaggle danh dau run FAILED.
import subprocess, sys, os, pathlib
REPO, DIR = 'https://github.com/HoangLong08/Paper-DAU.git', '/kaggle/working/qigoa'
if pathlib.Path(DIR, '.git').is_dir():
    subprocess.run(['git', '-C', DIR, 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', DIR, 'reset', '--hard', 'origin/main'], check=True)
else:
    subprocess.run(['git', 'clone', REPO, DIR], check=True)
COMMIT = subprocess.run(['git', '-C', DIR, 'rev-parse', 'HEAD'],
                        capture_output=True, text=True).stdout.strip()
print('git commit:', COMMIT)
NEEDED = ('scripts/run_ceiling.py', 'scripts/make_splits.py', 'configs/exp_ceiling.yaml',
          'configs/exp_main.yaml')
missing = [f for f in NEEDED if not pathlib.Path(DIR, f).exists()]
if missing:
    raise RuntimeError('THIEU FILE: %s -> repo chua dong bo. DUNG.' % (missing,))
os.chdir(DIR)
print('cwd =', os.getcwd())

def sh(args, fail_msg=''):
    args = [str(a) for a in args]
    print('\n$ ' + ' '.join(args), flush=True)
    rc = subprocess.run(args, cwd=DIR).returncode
    if rc != 0:
        raise RuntimeError('[FAIL rc=%d] %s\n%s' % (rc, ' '.join(args), fail_msg))
    return rc

sh([sys.executable, '-m', 'pip', 'install', '-q', 'medpy', 'tifffile'], 'medpy build loi.')

## Cell 2 - dataset auto-detect + symlink

In [ ]:
# GUARD + tu do duong dan BraTS (Kaggle mount tai /kaggle/input/datasets/awsaf49/...).
import itertools
inp = pathlib.Path('/kaggle/input')
if not inp.is_dir():
    raise RuntimeError('KHONG CO /kaggle/input -> chay ngoai Kaggle?')
print('Top-level /kaggle/input:', sorted(p.name for p in inp.iterdir()) or '(RONG)')
DEFAULT = pathlib.Path('/kaggle/input/brats20-dataset-training-validation/'
                       'BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData')
if DEFAULT.is_dir():
    REAL = DEFAULT
else:
    hits = [p for p in inp.rglob('MICCAI_BraTS2020_TrainingData') if p.is_dir()]
    REAL = hits[0] if hits else None
if REAL is None:
    raise RuntimeError('KHONG TIM THAY MICCAI_BraTS2020_TrainingData duoi /kaggle/input.\n'
                       'Top-level: ' + (', '.join(sorted(p.name for p in inp.iterdir())) or '(RONG)') + '\n'
                       'LUU Y: Kaggle chi luu Add Input khi ban bam Save Version.')
n_cases = sum(1 for p in REAL.iterdir() if p.is_dir())
print('Tim thay BraTS tai:', REAL, '| so ca:', n_cases)
if REAL != DEFAULT:
    link = pathlib.Path(DIR, 'data/brats20/MICCAI_BraTS2020_TrainingData')
    link.parent.mkdir(parents=True, exist_ok=True)
    if not (link.is_symlink() or link.exists()):
        link.symlink_to(REAL, target_is_directory=True)
    print('Da symlink root_local ->', REAL)

## Cell 3 - make_splits + run_ceiling --stage all

In [ ]:
# make_splits (folds cho P5, tranh WARN) + run_ceiling --stage all (oracle ladder + classical + P5).
sh([sys.executable, 'scripts/make_splits.py', '--config', 'configs/exp_ceiling.yaml', '--resume'],
   'make_splits FAIL.')
sh([sys.executable, 'scripts/run_ceiling.py', '--config', 'configs/exp_ceiling.yaml',
    '--stage', 'all', '--resume'],
   'run_ceiling --stage all FAIL (loi ky thuat, khac voi mot ket qua).')
print('\nOK -> results/ceiling/{raw,qstar,summary}.csv co oracle ladder + classical + P5.')

## Cell 4 - in thang tran + dong goi

In [ ]:
# In thang tran (loai C) + dong goi.
import pandas as pd, time, tarfile, glob, json
c = pd.read_csv('results/ceiling/summary.csv', comment='#')
print('=== CEILING SUMMARY (method_class + dice_median) ===')
print(c[['target','include_zero_bg','method','method_class','n_patients','dice_median']].to_string(index=False))
stamp = time.strftime('%Y%m%d_%H%M')
tgz = '/kaggle/working/ceiling_%s.tgz' % stamp
with tarfile.open(tgz, 'w:gz') as t:
    for d in ('results/ceiling', 'data/splits'):
        if pathlib.Path(d).is_dir():
            t.add(d)
print('\nDa dong goi:', tgz)
for m in sorted(glob.glob('results/ceiling/run-manifest.json')):
    print(m, '->', json.dumps(json.load(open(m))['extra'], ensure_ascii=False)[:300])
print('git commit:', COMMIT)